In [7]:
!pip install pyfpgrowth


In [8]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pyfpgrowth

Ta sẽ sử dụng fpgrowth để tạo luật kết hợp với dataset là eCommerce behavior data from multi category store (tháng 10)

In [9]:
DATA_PATH = "/kaggle/input/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv"


df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(42448764, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


Đảm bảo cột event chỉ trong 3 loại view, cart, purchase và các dòng luôn có product_id và user_session khác NaN

In [10]:
df = df[df['event_type'].isin(['view', 'cart', 'purchase'])]
df = df.dropna(subset=['user_session', 'product_id'])
print(df.shape)

(42448762, 9)


In [11]:
num_sessions = df['user_session'].nunique()
num_products = df['product_id'].nunique()


print("Sessions:", num_sessions)
print("Products:", num_products)

Sessions: 9244421
Products: 166794


Ta sẽ loại bỏ các session chỉ có 1 tương tác với 1 sản phẩm. 

Để thể hiện tính nguyên nhân -> kết quả, ta cần đồng thời xuất hiện cả nguyên nhân và kết quả. Do đó việc loại bỏ các session chỉ có 1 tương tác với 1 sản phẩm là hợp lý, vì khi đó ta chưa có ít nhất 2 phần tử để biểu diễn quan hệ nguyên nhân - kết quả trong 1 session

Việc lọc còn giúp giảm số session còn 1 nửa, giúp giảm độ lớn ma trận transaction khi tạo rule

In [12]:
# bỏ session chỉ xem 1 sản phẩm
session_sizes = df.groupby('user_session').size()
valid_sessions = session_sizes[session_sizes >= 2].index


df = df[df['user_session'].isin(valid_sessions)]
print("Remaining sessions:", df['user_session'].nunique())

Remaining sessions: 5974844


Tạo ma trận transaction

In [13]:
transactions = (
df.groupby('user_session')['product_id']
.apply(lambda x: list(set(x)))
.tolist()
)


print("#transactions:", len(transactions))
print(transactions[0])

#transactions: 5974844
[54900011]


Độ dài của transaction: 50% session có ≥3 sản phẩm, 10% session có ≥9 sản phẩm

In [14]:
import numpy as np
import pandas as pd

lengths = np.array([len(t) for t in transactions])

percentiles = [50, 75, 90, 95, 99]
values = np.percentile(lengths, percentiles)

pd.DataFrame({
    "percentile": percentiles,
    "transaction_length": values.astype(int)
})


,percentile,transaction_length
0,50,3
1,75,5
2,90,9
3,95,12
4,99,24


95% sản phẩm xuất hiện < 500 lần. Chỉ 1% sản phẩm xuất hiện > 2000 lần

In [15]:
from collections import Counter
import numpy as np
import pandas as pd

item_counter = Counter()
for t in transactions:
    item_counter.update(t)

supports = np.array(list(item_counter.values()))

percentiles = [50, 75, 90, 95, 99]
values = np.percentile(supports, percentiles)

pd.DataFrame({
    "percentile": percentiles,
    "item_support": values.astype(int)
})


,percentile,item_support
0,50,18
1,75,62
2,90,213
3,95,465
4,99,2055


Từ các thông tin về transaction, ta sẽ điều chỉnh support sao cho rule không quá chặt hay bị sai sót

In [ ]:
SUPPORT = 200

patterns = pyfpgrowth.find_frequent_patterns(
transactions,
support_threshold=SUPPORT
)

len(patterns)

70292

- Rule dạng: X → Y
- confidence = P(Y | X)

In [ ]:
CONFIDENCE = 0.3

rules = pyfpgrowth.generate_association_rules(
patterns,
confidence_threshold=CONFIDENCE
)

len(rules)

11164

- Chuyển rule sang dạng bảng để phân tích / lọc / visualize
- Thêm độ dài antecedent & consequent để đánh giá độ phức tạp của rule

In [ ]:
rule_list = []

for antecedent, (consequent, conf) in rules.items():
    rule_list.append({
    'antecedents': set(antecedent),
    'consequents': set(consequent),
    'confidence': conf,
    'len_antecedent': len(antecedent),
    'len_consequent': len(consequent)
    })

rules_df = pd.DataFrame(rule_list)
rules_df.sort_values(by='confidence', ascending=False).head()


,antecedents,consequents,confidence,len_antecedent,len_consequent
954,"{10800025, 10800074, 10800180}",{10800172},0.895652,3,1
1033,"{10800025, 10800074, 10800076, 10800182}",{10800172},0.882784,4,1
1534,"{10800025, 10800074, 10800076, 10800132}",{10800172},0.868217,4,1
707,"{10800025, 10800074, 10800068}",{10800172},0.867188,3,1
1025,"{10800048, 10800025, 10800076, 10800182}",{10800172},0.864198,4,1


In [ ]:
print(f"Min len_antecedent: {rules_df['len_antecedent'].min()}")
print(f"Max len_antecedent: {rules_df['len_antecedent'].max()}")
print(f"Min len_consequent: {rules_df['len_consequent'].min()}")
print(f"Max len_consequent: {rules_df['len_consequent'].max()}")

Min len_antecedent: 1
Max len_antecedent: 5
Min len_consequent: 1
Max len_consequent: 1


In [34]:
rules_df.head(10)

,antecedents,consequents,confidence,len_antecedent,len_consequent
0,{16500035},{16500007},0.601173,1,1
1,{25900028},{25900012},0.582133,1,1
2,{6301345},{6302016},0.559889,1,1
3,{1701552},{1701474},0.602180,1,1
4,{7300205},{7300440},0.596257,1,1
5,{11800026},{11800010},0.574074,1,1
6,{1480762},{1480492},0.547368,1,1
7,{2501010},{2501143},0.685714,1,1
8,{27300018},{27300001},0.547315,1,1
9,{18700007},{18700008},0.505051,1,1


- Input: danh sách sản phẩm người dùng đã xem/mua (viewed_products)

- Cách hoạt động:
     - Lọc các rule có antecedent ⊆ viewed_products
     - Ưu tiên rule có confidence cao
     - Gộp consequents, loại trùng & tránh gợi ý lại sản phẩm đã xem

In [ ]:
def recommend_products(viewed_products, rules_df, top_k=5):
    viewed_products = set(viewed_products)

    candidates = rules_df[
    rules_df['antecedents'].apply(lambda x: x.issubset(viewed_products))
    ].sort_values(by='confidence', ascending=False)
    
    recs = []
    seen = set(viewed_products)
    
    for _, row in candidates.iterrows():
        for p in row['consequents']:
            if p not in seen:
                recs.append(p)
                seen.add(p)
            if len(recs) >= top_k:
                return recs
    return recs

Dựa vào rule_df, ta sẽ recommend với 1 ví dụ

In [ ]:
example_session = [10800025, 10800074, 10800180]

In [ ]:
recs = recommend_products(
    viewed_products=example_session,
    rules_df=rules_df,
    top_k=5
)
recs

[10800172]

Để có được insight về mối liên hệ giữa sản phẩm được gợi ý với sản phẩm mà người dùng tương tác, ta sẽ truy ngược lại các giá trị category_code, brand và price của các product_id mà người dùng tương tác và luật kết hợp gợi ý

In [56]:
product_meta = (
    df[['product_id', 'category_code', 'brand', 'price']]
    .drop_duplicates('product_id')
    .set_index('product_id')
)

In [ ]:
def explain_session(session, recs, product_meta):
    session_df = product_meta.loc[session]
    rec_df = product_meta.loc[recs]
    return session_df, rec_df

In [ ]:
session_df, rec_df = explain_session(
    example_session, recs, product_meta
)

Ví dụ với người dùng tương tác tác với 3 sản phẩm 10800025, 10800074, 10800180

In [ ]:
session_df

,category_code,brand,price
product_id,,,
10800025,NaN,redmond,56.60
10800074,NaN,redmond,41.16
10800180,NaN,redmond,84.92


Qua đó ta thấy được sự liên hệ về sản phảm được gợi ý với 3 sản phẩm trước đó là cùng hãng redmond 

In [60]:
rec_df

,category_code,brand,price
product_id,,,
10800172,NaN,redmond,61.75


Tuy nhiên 32% của cột category_code là NaN để đảm bảo đưa ra thông tin đầy đủ, ta sẽ lọc ra các antecedent (product_id) mà không bị NaN trong rule_df như sau

In [71]:
rule_products = set().union(
    *rules_df['antecedents'],
    *rules_df['consequents']
)

In [61]:
product_category = (
    df[['product_id', 'category_code']]
    .drop_duplicates('product_id')
    .set_index('product_id')
)

In [70]:
valid_products = set(
    product_category[product_category['category_code'].notna()].index
)


Lấy giao của  rule_product (chứa các product_id có NaN ở category_code) với valid_products (chứa các product_id không có NaN ở category_code)

In [ ]:
rule_valid_products = rule_products & valid_products

Ta sẽ tạo ví dụ mới với các product id đảm bảo không bị NaN ở category code

In [ ]:
example_session = list(rule_valid_products)
example_session = example_session[:10]
example_session

[3801134,
 3801135,
 2900022,
 11100223,
 21405784,
 21405807,
 2900087,
 8700025,
 2900090,
 11100287]

In [86]:
recs = recommend_products(
    viewed_products=example_session,
    rules_df=rules_df,
    top_k=5
)
recs

[2900569, 8700289]

In [ ]:
session_df, rec_df = explain_session(
    example_session, recs, product_meta
)

In [88]:
session_df

,category_code,brand,price
product_id,,,
3801134,appliances.iron,elenberg,16.71
3801135,appliances.iron,elenberg,20.57
2900022,appliances.kitchen.microwave,samsung,126.10
11100223,appliances.personal.scales,elenberg,14.13
21405784,electronics.clocks,casio,183.02
21405807,electronics.clocks,casio,246.85
2900087,appliances.kitchen.microwave,panasonic,92.64
8700025,appliances.personal.hair_cutter,remington,40.64
2900090,appliances.kitchen.microwave,samsung,102.73


In [82]:
rec_df

,category_code,brand,price
product_id,,,
2900569,appliances.kitchen.microwave,lg,95.01
8700289,appliances.personal.hair_cutter,elenberg,23.14


Qua các thông tin trên ta đã có được liên hệ giữa gợi ý và các sản phẩm mà người dùng tương tác, chúng có thể là cùng 1 loại appliances.kitchen hay appliances.personal, ngoài ra chúng còn có thể cùng hãng elenberg hoặc cùng 1 lượng giá phù hợp